# 🧠 ImpactAI — Data Analysis & ML Model Training

This notebook provides comprehensive visualization of the mental health counseling dataset
used by ImpactAI, and trains an ML severity classifier.

**Sections:**
1. Data Loading & Overview
2. Text Length Analysis
3. Word Frequency Analysis
4. Topic/Keyword Analysis
5. Severity Label Distribution
6. ML Model Training & Evaluation
7. Model Performance Visualization

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# Styling
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

print('✅ All imports successful!')

## 1️⃣ Data Loading & Overview

In [ ]:
# Load the dataset
df = pd.read_csv('data/train.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nTotal rows: {len(df):,}')
print(f'Missing Context: {df["Context"].isna().sum()}')
print(f'Missing Response: {df["Response"].isna().sum()}')
print(f'\n--- First 3 rows ---')
df.head(3)

In [ ]:
# Basic statistics
df['context_words'] = df['Context'].astype(str).str.split().str.len()
df['response_words'] = df['Response'].astype(str).str.split().str.len()
df['context_chars'] = df['Context'].astype(str).str.len()
df['response_chars'] = df['Response'].astype(str).str.len()

print('📊 Text Length Statistics (word count):')
print('=' * 50)
stats = pd.DataFrame({
    'Context': df['context_words'].describe(),
    'Response': df['response_words'].describe()
}).round(2)
stats

## 2️⃣ Text Length Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Context word count distribution
axes[0, 0].hist(df['context_words'], bins=40, color='#4f46e5', edgecolor='white', alpha=0.85)
axes[0, 0].set_title('Context — Word Count Distribution')
axes[0, 0].set_xlabel('Words')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['context_words'].mean(), color='red', linestyle='--', label=f'Mean: {df["context_words"].mean():.0f}')
axes[0, 0].legend()

# Response word count distribution
axes[0, 1].hist(df['response_words'], bins=40, color='#14b8a6', edgecolor='white', alpha=0.85)
axes[0, 1].set_title('Response — Word Count Distribution')
axes[0, 1].set_xlabel('Words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df['response_words'].mean(), color='red', linestyle='--', label=f'Mean: {df["response_words"].mean():.0f}')
axes[0, 1].legend()

# Box plots
axes[1, 0].boxplot([df['context_words'], df['response_words']], labels=['Context', 'Response'],
                    patch_artist=True, boxprops=dict(facecolor='#818cf8', alpha=0.7))
axes[1, 0].set_title('Word Count — Box Plot Comparison')
axes[1, 0].set_ylabel('Word Count')

# Context vs Response scatter
sample = df.sample(min(1000, len(df)), random_state=42)
axes[1, 1].scatter(sample['context_words'], sample['response_words'],
                    alpha=0.4, color='#6366f1', edgecolors='white', linewidth=0.5)
axes[1, 1].set_title('Context vs Response Length (sample)')
axes[1, 1].set_xlabel('Context Words')
axes[1, 1].set_ylabel('Response Words')

plt.tight_layout()
plt.savefig('plots/text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Plot saved → plots/text_length_analysis.png')

## 3️⃣ Word Frequency Analysis

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Custom stop words for mental health domain
extra_stops = {'like', 'just', 'know', 'think', 'really', 'want', 'going',
               'get', 'make', 'feel', 'say', 'go', 'thing', 'time', 'way',
               'people', 'one', 'would', 'also', 'may', 'need', 'can',
               'help', 'try', 'new', 'us', 'said', 'could', 'much'}
stop_words = ENGLISH_STOP_WORDS.union(extra_stops)

def get_top_words(text_series, n=25):
    """Get top-n most frequent words (excluding stop words)."""
    all_words = []
    for text in text_series.astype(str):
        words = re.findall(r'\b[a-z]{3,}\b', text.lower())
        all_words.extend([w for w in words if w not in stop_words])
    return Counter(all_words).most_common(n)

context_words = get_top_words(df['Context'])
response_words = get_top_words(df['Response'])

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Context top words
words_c, counts_c = zip(*context_words)
axes[0].barh(range(len(words_c)), counts_c, color='#4f46e5', edgecolor='white')
axes[0].set_yticks(range(len(words_c)))
axes[0].set_yticklabels(words_c)
axes[0].invert_yaxis()
axes[0].set_title('Top 25 Words in User Questions (Context)')
axes[0].set_xlabel('Frequency')

# Response top words
words_r, counts_r = zip(*response_words)
axes[1].barh(range(len(words_r)), counts_r, color='#14b8a6', edgecolor='white')
axes[1].set_yticks(range(len(words_r)))
axes[1].set_yticklabels(words_r)
axes[1].invert_yaxis()
axes[1].set_title('Top 25 Words in Counselor Responses')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('plots/word_frequency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Plot saved → plots/word_frequency_analysis.png')

## 4️⃣ Mental Health Topic Analysis

In [ ]:
# Define mental health topic keywords
topics = {
    'Depression': ['depress', 'sad', 'hopeless', 'worthless', 'crying', 'empty'],
    'Anxiety': ['anxi', 'panic', 'worry', 'nervous', 'fear', 'phobia'],
    'Relationships': ['relationship', 'marriage', 'divorce', 'partner', 'spouse', 'husband', 'wife'],
    'Self-Esteem': ['self esteem', 'self-esteem', 'confidence', 'self worth', 'self-worth', 'insecure'],
    'Trauma/Abuse': ['trauma', 'abuse', 'ptsd', 'assault', 'violence', 'neglect'],
    'Sleep Issues': ['sleep', 'insomnia', 'nightmare', 'cant sleep', 'rest'],
    'Grief/Loss': ['grief', 'loss', 'death', 'mourning', 'passed away', 'died'],
    'Anger': ['anger', 'angry', 'rage', 'frustrat', 'irritab'],
    'Self-Harm': ['self harm', 'self-harm', 'cutting', 'hurt myself', 'harm myself'],
    'Substance Use': ['alcohol', 'drug', 'addiction', 'substance', 'drinking', 'sober'],
}

topic_counts = {}
for topic, keywords in topics.items():
    pattern = '|'.join(keywords)
    count = df['Context'].astype(str).str.lower().str.contains(pattern, regex=True).sum()
    topic_counts[topic] = count

topic_df = pd.DataFrame(list(topic_counts.items()), columns=['Topic', 'Count'])
topic_df = topic_df.sort_values('Count', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('viridis', len(topic_df))
bars = ax.barh(topic_df['Topic'], topic_df['Count'], color=colors, edgecolor='white')

# Add value labels
for bar, count in zip(bars, topic_df['Count']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{count}', va='center', fontweight='bold', fontsize=11)

ax.set_title('Mental Health Topics in User Questions', fontsize=16)
ax.set_xlabel('Number of Questions')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('plots/topic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Plot saved → plots/topic_analysis.png')

## 5️⃣ Severity Label Assignment & Distribution

In [ ]:
# --- Severity keyword patterns ---
CRISIS_PATTERNS = [
    r'\bsuicid\w*\b', r'\bkill\s+(my|your)self\b', r'\bself[- ]?harm\b',
    r'\bending\s+(my|it|life)\b', r"\bdon'?t\s+want\s+to\s+live\b",
    r'\bwant\s+to\s+die\b', r'\b988\b', r'\bcrisis\s+line\b', r'\bemergency\b',
]
HIGH_PATTERNS = [
    r'\bsexual\s+abuse\b', r'\babuse\b', r'\btrauma\b', r'\bPTSD\b',
    r'\bworthless\b', r'\bhopeless\b', r'\bpanic\s+attack\b',
    r'\bsevere\s+(depression|anxiety)\b', r"\bcan'?t\s+stop\s+crying\b",
    r'\bcut(ting)?\s+(my|your)self\b', r'\bhurt\s+(my|your)self\b',
]
MEDIUM_PATTERNS = [
    r'\bdepress(ed|ion)?\b', r'\banxi(ous|ety)\b', r'\bstress(ed)?\b',
    r'\boverwhelm(ed|ing)?\b', r'\binsomnia\b', r"\bcan'?t\s+sleep\b",
    r'\blow\s+self[- ]?esteem\b', r'\bsad(ness)?\b', r'\blonely\b',
    r'\bgrief\b', r'\bloss\b', r'\bdivorce\b', r'\brelationship\s+problem\b',
]

def assign_severity(text):
    text_lower = str(text).lower()
    for p in CRISIS_PATTERNS:
        if re.search(p, text_lower): return 'crisis'
    for p in HIGH_PATTERNS:
        if re.search(p, text_lower): return 'high'
    for p in MEDIUM_PATTERNS:
        if re.search(p, text_lower): return 'medium'
    return 'low'

# Combined text for richer features
df['text'] = df['Context'].astype(str) + ' ' + df['Response'].astype(str)
df['severity'] = df['text'].apply(assign_severity)

label_counts = df['severity'].value_counts()
print('📊 Severity Label Distribution:')
print('=' * 40)
for label, count in label_counts.items():
    pct = count / len(df) * 100
    print(f'  {label:>8s}: {count:>5,}  ({pct:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

severity_colors = {'low': '#22c55e', 'medium': '#eab308', 'high': '#f97316', 'crisis': '#ef4444'}
order = ['low', 'medium', 'high', 'crisis']
ordered_counts = label_counts.reindex(order).fillna(0).astype(int)

# Bar chart
bars = axes[0].bar(ordered_counts.index, ordered_counts.values,
                    color=[severity_colors[s] for s in ordered_counts.index],
                    edgecolor='white', linewidth=2)
for bar, val in zip(bars, ordered_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Severity Level Distribution')
axes[0].set_ylabel('Number of Examples')
axes[0].set_xlabel('Severity Level')

# Pie chart
axes[1].pie(ordered_counts.values, labels=ordered_counts.index,
            colors=[severity_colors[s] for s in ordered_counts.index],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Severity Level Proportions')

plt.tight_layout()
plt.savefig('plots/severity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Plot saved → plots/severity_distribution.png')

## 6️⃣ ML Model Training

In [ ]:
print('🔧 Training TF-IDF + Logistic Regression Pipeline')
print('=' * 55)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(df['text'])
y = df['severity']

print(f'Feature matrix shape: {X.shape}')
print(f'Labels: {list(y.unique())}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain size: {X_train.shape[0]:,}')
print(f'Test size:  {X_test.shape[0]:,}')

# Train model
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=1.0,
    solver='lbfgs',
    multi_class='multinomial',
    random_state=42,
)
model.fit(X_train, y_train)

# 5-fold cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'\n📊 5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'   Per-fold: {[f"{s:.4f}" for s in cv_scores]}')

In [ ]:
# Evaluate on test set
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'\n🎯 Test Accuracy: {accuracy:.4f}')
print(f'\n{"=" * 55}')
print(classification_report(y_test, y_pred, zero_division=0))

## 7️⃣ Model Performance Visualization

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(cm, display_labels=model.classes_)
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized confusion matrix
cm_norm = confusion_matrix(y_test, y_pred, labels=model.classes_, normalize='true')
disp_norm = ConfusionMatrixDisplay(cm_norm, display_labels=model.classes_)
disp_norm.plot(ax=axes[1], cmap='Oranges', values_format='.2f')
axes[1].set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Plot saved → plots/confusion_matrices.png')

In [ ]:
# Per-class metrics bar chart
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
classes = [c for c in model.classes_ if c in report]
metrics_df = pd.DataFrame({
    'Precision': [report[c]['precision'] for c in classes],
    'Recall': [report[c]['recall'] for c in classes],
    'F1-Score': [report[c]['f1-score'] for c in classes],
}, index=classes)

fig, ax = plt.subplots(figsize=(12, 6))
metrics_df.plot(kind='bar', ax=ax, color=['#4f46e5', '#14b8a6', '#f59e0b'],
                edgecolor='white', linewidth=1.5)
ax.set_title('Per-Class Precision / Recall / F1-Score')
ax.set_ylabel('Score')
ax.set_xlabel('Severity Class')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.tick_params(axis='x', rotation=0)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=9, padding=2)

plt.tight_layout()
plt.savefig('plots/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Plot saved → plots/per_class_metrics.png')

In [ ]:
# Cross-validation scores visualization
fig, ax = plt.subplots(figsize=(10, 5))

folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
colors = ['#4f46e5' if s >= cv_scores.mean() else '#f97316' for s in cv_scores]
bars = ax.bar(folds, cv_scores, color=colors, edgecolor='white', linewidth=2)

ax.axhline(cv_scores.mean(), color='#ef4444', linestyle='--', linewidth=2,
           label=f'Mean: {cv_scores.mean():.4f}')
ax.fill_between(range(len(folds)),
                cv_scores.mean() - cv_scores.std(),
                cv_scores.mean() + cv_scores.std(),
                alpha=0.15, color='red', label=f'±1 Std: {cv_scores.std():.4f}')

for bar, val in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold')

ax.set_title('5-Fold Cross-Validation Accuracy')
ax.set_ylabel('Accuracy')
ax.set_ylim(min(cv_scores) - 0.05, max(cv_scores) + 0.05)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('plots/cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Plot saved → plots/cross_validation.png')

In [ ]:
# Top features per severity class
feature_names = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
class_colors = {'low': '#22c55e', 'medium': '#eab308', 'high': '#f97316', 'crisis': '#ef4444'}

for idx, (cls, ax) in enumerate(zip(model.classes_, axes.flat)):
    cls_idx = list(model.classes_).index(cls)
    coefs = model.coef_[cls_idx]
    top_indices = np.argsort(coefs)[-15:]  # top 15 features
    top_features = [feature_names[i] for i in top_indices]
    top_coefs = coefs[top_indices]
    
    ax.barh(top_features, top_coefs, color=class_colors.get(cls, '#6366f1'),
            edgecolor='white')
    ax.set_title(f'Top Features — {cls.upper()}')
    ax.set_xlabel('Coefficient Weight')

plt.tight_layout()
plt.savefig('plots/top_features_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Plot saved → plots/top_features_per_class.png')

## 💾 Save Trained Model

In [ ]:
import pickle
from pathlib import Path

model_dir = Path('ml_models')
model_dir.mkdir(exist_ok=True)

with open(model_dir / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open(model_dir / 'severity_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('✅ Model saved to ml_models/')
print(f'   Vectorizer: {(model_dir / "tfidf_vectorizer.pkl").stat().st_size / 1024:.1f} KB')
print(f'   Model:      {(model_dir / "severity_model.pkl").stat().st_size / 1024:.1f} KB')

## 🧪 Test Predictions

In [ ]:
# Test with sample messages
test_messages = [
    "I feel happy and grateful today!",
    "I'm stressed about my upcoming exams and can't concentrate.",
    "I've been feeling really depressed and hopeless for weeks.",
    "I have been through sexual abuse and I can't cope anymore.",
    "I want to end my life. I don't see any point in living.",
    "My relationship with my partner has been causing me anxiety.",
    "I can't sleep at night and feel worthless every single day.",
    "How can I manage my stress better during the work week?",
]

print('🧪 Test Predictions')
print('=' * 80)
for msg in test_messages:
    X_test_msg = vectorizer.transform([msg])
    pred = model.predict(X_test_msg)[0]
    probas = model.predict_proba(X_test_msg)[0]
    confidence = max(probas)
    
    severity_emoji = {'low': '🟢', 'medium': '🟡', 'high': '🟠', 'crisis': '🔴'}
    print(f'\n{severity_emoji.get(pred, "⚪")} [{pred.upper():>7s}] (conf: {confidence:.2f})')
    print(f'   "{msg[:70]}..."' if len(msg) > 70 else f'   "{msg}"')

## 📊 Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('ImpactAI — Data & Model Summary Dashboard', fontsize=18, fontweight='bold', y=1.02)

# 1. Dataset size
axes[0, 0].text(0.5, 0.5, f'{len(df):,}', fontsize=48, fontweight='bold',
                ha='center', va='center', color='#4f46e5',
                transform=axes[0, 0].transAxes)
axes[0, 0].text(0.5, 0.2, 'Total Training Examples', fontsize=14,
                ha='center', va='center', color='gray',
                transform=axes[0, 0].transAxes)
axes[0, 0].axis('off')

# 2. Model accuracy
axes[0, 1].text(0.5, 0.5, f'{accuracy:.1%}', fontsize=48, fontweight='bold',
                ha='center', va='center', color='#14b8a6',
                transform=axes[0, 1].transAxes)
axes[0, 1].text(0.5, 0.2, 'Test Accuracy', fontsize=14,
                ha='center', va='center', color='gray',
                transform=axes[0, 1].transAxes)
axes[0, 1].axis('off')

# 3. Number of features
axes[0, 2].text(0.5, 0.5, f'{X.shape[1]:,}', fontsize=48, fontweight='bold',
                ha='center', va='center', color='#f59e0b',
                transform=axes[0, 2].transAxes)
axes[0, 2].text(0.5, 0.2, 'TF-IDF Features', fontsize=14,
                ha='center', va='center', color='gray',
                transform=axes[0, 2].transAxes)
axes[0, 2].axis('off')

# 4. Severity distribution
ordered_counts.plot(kind='bar', ax=axes[1, 0],
                     color=[severity_colors[s] for s in ordered_counts.index],
                     edgecolor='white')
axes[1, 0].set_title('Severity Distribution')
axes[1, 0].tick_params(axis='x', rotation=0)

# 5. Word count distributions
axes[1, 1].hist(df['context_words'], bins=30, alpha=0.7, color='#4f46e5', label='Context')
axes[1, 1].hist(df['response_words'], bins=30, alpha=0.5, color='#14b8a6', label='Response')
axes[1, 1].set_title('Word Count Distributions')
axes[1, 1].legend()

# 6. CV scores
axes[1, 2].bar(folds, cv_scores, color='#6366f1', edgecolor='white')
axes[1, 2].axhline(cv_scores.mean(), color='red', linestyle='--')
axes[1, 2].set_title('Cross-Validation Scores')
axes[1, 2].set_ylim(min(cv_scores) - 0.05, max(cv_scores) + 0.05)

plt.tight_layout()
plt.savefig('plots/summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ All visualizations complete!')
print('📂 Saved plots in the plots/ directory')